# ATML PA3 — Task 1: GSM8K at batch size 32 for Full FT (mid), Full FT and L2-SP, then full analysis

The first evaluations ran GSM8K at batch size 1 for baseline, Full FT (mid), Full FT and L2-SP, and at 32
for the LoRA runs. This reruns the three Full FT conditions at 32 (one per GPU), takes the baseline at 32
from the earlier batch check, and keeps every batch-size-1 run as `<condition>/gsm8k_bs1`. Then it
redoes the whole Task 1 analysis on the consistent results.

GPU T4 x2, Internet on. Attach `atml-pa3-code` (latest version) **and** the finished output of the
Task 1 finish notebook (Add Input → Your Work → Notebooks). *Save & Run All (Commit)*: about 1.5 h.
Download `task1_analysis.zip` from the Output tab.

In [ ]:
# --- locate code (Kaggle dataset "atml-pa3-code" containing atml-pa3-code.zip, or Colab Drive) ---
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content/drive/MyDrive/atml-pa3"
if not ON_KAGGLE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
REPO = "/tmp/ATML-PA3"           # code lives outside the saved output
zips = glob.glob("/kaggle/input/**/atml-pa3-code.zip", recursive=True) + glob.glob(f"{WORK}/atml-pa3-code.zip")
# Kaggle usually unpacks an uploaded zip, so also accept an already-extracted copy
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/task1/train.py", recursive=True)]
shutil.rmtree(REPO, ignore_errors=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    assert dirs, "upload outputs/atml-pa3-code.zip as a Kaggle dataset (or to Drive/atml-pa3/)"
    shutil.copytree(dirs[0], REPO)
print("code from", zips[:1] or dirs[:1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "hf_xet"], check=True)
os.chdir(REPO)
os.makedirs(f"{WORK}/outputs", exist_ok=True)
if not os.path.exists("outputs"):
    os.symlink(f"{WORK}/outputs", "outputs")
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import json, random, re, threading, time
import pandas as pd
CONDS = ["baseline", "full_u625", "full", "l2sp", "lora4", "lora16", "oplora16"]
RERUN = ["full_u625", "full", "l2sp"]
roots = glob.glob("/kaggle/input/**/outputs/results_task1", recursive=True)
assert roots, "attach the finish notebook's output: Add Input -> Your Work -> Notebooks"
SRC = max(roots, key=lambda r: sum(os.path.exists(f"{r}/{c}/qualitative.json") for c in CONDS))
TRAIN = os.path.join(os.path.dirname(SRC), "task1")
RES, TMP, OUT = "outputs/results_task1", "outputs/gsm8k_bs32", "outputs/analysis_task1"
MODELS = "/kaggle/temp/eval_models" if ON_KAGGLE else "/content/eval_models"
if not os.path.exists(RES):
    shutil.copytree(SRC, RES)       # writable copy; the reruns are swapped into it below
os.makedirs(TMP, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("results from:", SRC, "\ntraining:", TRAIN)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lm_eval==0.4.13", "langdetect", "immutabledict"], check=True)

In [ ]:
todo = [c for c in RERUN if not os.path.exists(f"{RES}/{c}/gsm8k_bs1")]
status = {}
if todo:
    subprocess.run([sys.executable, "-m", "task1.prepare_eval", "--runs", TRAIN, "--out", MODELS, "--only", *todo], check=True)
    n_gpu = max(torch.cuda.device_count(), 1)

    def worker(gpu, conds):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
        for c in conds:
            with open(f"{TMP}/{c}.log", "w") as log:
                status[c] = subprocess.run([sys.executable, "-m", "task1.evaluate", "--backend", "hf", "--models", MODELS,
                                            "--out", TMP, "--conditions", c, "--skip", "lik", "qual"],
                                           env=env, stdout=log, stderr=subprocess.STDOUT).returncode

    threads = [threading.Thread(target=worker, args=(g, todo[g::n_gpu])) for g in range(n_gpu)]
    for t in threads:
        t.start()
    while any(t.is_alive() for t in threads):
        time.sleep(300)
        for c in todo:
            if os.path.exists(f"{TMP}/{c}.log"):
                tail = open(f"{TMP}/{c}.log").read().replace("\r", "\n").strip().splitlines()[-1:]
                print(time.strftime("%H:%M"), c, tail, flush=True)
    print("exit codes:", status)
    shutil.rmtree(MODELS, ignore_errors=True)

In [ ]:
# swap the batch-32 runs in; batch-size-1 runs are kept beside them as gsm8k_bs1
for c in RERUN:
    new = f"{TMP}/{c}/gsm8k"
    if os.path.exists(f"{RES}/{c}/gsm8k_bs1"):
        continue
    assert status.get(c) == 0 and glob.glob(f"{new}/**/results_*.json", recursive=True), f"rerun of {c} failed: see {TMP}/{c}.log"
    shutil.move(f"{RES}/{c}/gsm8k", f"{RES}/{c}/gsm8k_bs1")
    shutil.move(new, f"{RES}/{c}/gsm8k")
if not os.path.exists(f"{RES}/baseline/gsm8k_bs1"):
    shutil.move(f"{RES}/baseline/gsm8k", f"{RES}/baseline/gsm8k_bs1")
    shutil.copytree(f"{RES}/_batchcheck/baseline/gsm8k", f"{RES}/baseline/gsm8k")
for c in CONDS:
    f = sorted(glob.glob(f"{RES}/{c}/gsm8k/**/results_*.json", recursive=True))[-1]
    r = json.load(open(f))
    print(c, "batch", r["config"]["batch_size"], "flex", round(r["results"]["gsm8k_cot"]["exact_match,flexible-extract"], 4))
    assert str(r["config"]["batch_size"]) == "32", c

In [ ]:
# benchmark table, per-example GSM8K files, learning curves, resources, qualitative sheets
subprocess.run([sys.executable, "-m", "task1.analyze", "--runs", TRAIN, "--eval", RES, "--out", OUT], check=True)

In [ ]:
# weight-space (rho_k, eta) and activation (R) diagnostics on down_proj of layers 0, 13, 27
subprocess.run([sys.executable, "-m", "task1.diagnostics", "--runs", TRAIN, "--out", f"{OUT}/diagnostics"], check=True)

In [ ]:
# Does GSM8K's flexible extraction favour one answer style? Stock scores stay the primary metric.
# Heuristic rescoring: use the response's last \boxed{} (else a bold number near its end) when it
# has one and was not truncated, otherwise the stock extraction; compare numbers, not strings.
def num(s):
    s = re.sub(r"\\text\{[^}]*\}|\\[,!]|[$,%*\\]", "", str(s))
    m = re.findall(r"-?\d+(?:\.\d+)?", s)
    return float(m[-1]) if m else None

def boxed(t):
    i = t.rfind("\\boxed{")
    if i < 0:
        return None
    depth, j = 1, i + 7
    while j < len(t) and depth:
        depth += {"{": 1, "}": -1}.get(t[j], 0)
        j += 1
    return t[i + 7:j - 1] if depth == 0 else None

def final_answer(t):
    b = boxed(t)
    if b is not None and num(b) is not None:
        return num(b)
    for b in reversed(re.findall(r"\*\*(.+?)\*\*", t[-300:], re.S)):
        if num(b) is not None:
            return num(b)
    return None

def same(a, b):
    return a is not None and b is not None and abs(a - b) < 1e-6

rows, frames = [], {}
for c in CONDS:
    g = pd.read_csv(f"{OUT}/gsm8k_per_example_{c}.csv", dtype={"extracted": str, "target": str}, keep_default_na=False)
    final = [None if tr else final_answer(t) for t, tr in zip(g.response, g.truncated)]
    g["final"] = final
    g["alt"] = [f if f is not None else num(e) for f, e in zip(final, g.extracted)]
    g["alt_correct"] = [float(same(a, num(t))) for a, t in zip(g.alt, g.target)]
    frames[c] = g
    rows.append({"condition": c, "stock_%": 100 * g.correct.mean(), "rescored_%": 100 * g.alt_correct.mean(),
                 "wrong_to_right": int(((g.correct == 0) & (g.alt_correct == 1)).sum()),
                 "right_to_wrong": int(((g.correct == 1) & (g.alt_correct == 0)).sum()),
                 "has_final_marker": int(g.final.notna().sum())})
E = pd.DataFrame(rows)
for k in ("stock_%", "rescored_%"):
    E[k + "_change"] = E[k] - E[k].iloc[0]
E.to_csv(f"{OUT}/gsm8k_extraction_check.csv", index=False)
print(E.round(2).to_string())

# sheet for reading by hand: is the heuristic right, and what did it miss on the baseline?
b = frames["baseline"]
flips = b[(b.correct == 0) & (b.alt_correct == 1)]
wrong = b[(b.correct == 0) & (b.alt_correct == 0)]
picked = wrong.sample(min(20, len(wrong)), random_state=6304)
md = []
for title, part in (("Baseline: stock wrong, rescored right", flips), ("Baseline: wrong under both (random 20)", picked)):
    md.append(f"\n## {title} ({len(part)})\n")
    for _, r in part.iterrows():
        md.append(f"**doc {r.doc_id}** gold {r.target}, stock {r.extracted}, rescored {r.alt}\n\n```\n...{r.response[-250:]}\n```\n")
open(f"{OUT}/gsm8k_extraction_review_baseline.md", "w", encoding="utf-8").write("\n".join(md))
for _, r in flips.head(5).iterrows():
    print(f"\ndoc {r.doc_id}: gold {r.target}, stock {r.extracted}, rescored {r.alt}\n...{r.response[-150:]}")

In [ ]:
# GSM8K batch size per condition, and what batch size 1 vs 32 changed where both runs exist
def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    return files[-1] if files else None

def flex_samples(path):
    out = {}
    for line in open(path, encoding="utf-8"):
        x = json.loads(line)
        if x.get("filter") == "flexible-extract":
            out[x["doc_id"]] = (x["resps"][0][0] if isinstance(x["resps"][0], list) else x["resps"][0],
                                float(x["exact_match"]))
    return out

batch = {"batch_size": {}, "bs1_vs_bs32": {}}
for c in CONDS:
    r = json.load(open(latest(f"{RES}/{c}/gsm8k/**/results_*.json")))
    batch["batch_size"][c] = r.get("config", {}).get("batch_size")
    cur = latest(f"{RES}/{c}/gsm8k/**/samples_gsm8k_cot_*.jsonl")
    bs1 = latest(f"{RES}/{c}/gsm8k_bs1/**/samples_gsm8k_cot_*.jsonl")
    chk = latest(f"{RES}/_batchcheck/{c}/gsm8k/**/samples_gsm8k_cot_*.jsonl")
    old, new = (bs1, cur) if bs1 else (cur, chk) if chk else (None, None)
    if old:
        a, b = flex_samples(old), flex_samples(new)
        batch["bs1_vs_bs32"][c] = {
            "n": len(b), "bs1_%": 100 * sum(v[1] for v in a.values()) / len(a),
            "bs32_%": 100 * sum(v[1] for v in b.values()) / len(b),
            "identical_responses": sum(a[d][0] == b[d][0] for d in b),
            "wrong_to_right": sum(a[d][1] == 0 and b[d][1] == 1 for d in b),
            "right_to_wrong": sum(a[d][1] == 1 and b[d][1] == 0 for d in b)}
json.dump(batch, open(f"{OUT}/gsm8k_batch_settings.json", "w"), indent=1, default=int)
print(json.dumps(batch, indent=1, default=int))

In [ ]:
# benchmark changes from baseline: one panel per benchmark on a common scale
import matplotlib.pyplot as plt
T = pd.read_csv(f"{OUT}/benchmarks.csv").set_index("condition")
order = ["full_u625", "full", "l2sp", "lora4", "lora16", "oplora16"]
panels = ["GSM8K (flex EM)", "HellaSwag acc_norm", "BoolQ acc", "ARC-E acc_norm"]
fig, axes = plt.subplots(1, 4, figsize=(11, 3), sharey=True, sharex=True)
for ax, p in zip(axes, panels):
    v = T.loc[order, p + " Δpp"]
    ax.barh(range(len(order)), v, height=0.55, color="#2a78d6")
    for i, x in enumerate(v):
        ax.text(x + (0.6 if x >= 0 else -0.6), i, f"{x:+.1f}", va="center", ha="left" if x >= 0 else "right", fontsize=8, color="#333333")
    ax.axvline(0, color="#555555", lw=0.8)
    ax.set_xlim(-12, 30)
    ax.set_title(p, fontsize=10, loc="left")
    ax.set_xlabel("change from baseline (pp)", fontsize=8)
    ax.tick_params(length=0, labelsize=8)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
axes[0].set_yticks(range(len(order)), [T.loc[c, "label"] for c in order])
axes[0].invert_yaxis()
fig.tight_layout()
fig.savefig(f"{OUT}/benchmark_changes.png", dpi=200)
plt.show()

In [ ]:
# one file to download
shutil.make_archive(f"{WORK}/task1_analysis", "zip", OUT)
print(f"{WORK}/task1_analysis.zip", os.path.getsize(f"{WORK}/task1_analysis.zip") // 1024, "KB")
for f in sorted(glob.glob(f"{OUT}/**", recursive=True)):
    if os.path.isfile(f):
        print(os.path.getsize(f) // 1024, "KB", os.path.relpath(f, OUT))
print(open(f"{OUT}/benchmarks.md", encoding="utf-8").read())